# Day 4-02｜人體姿態點與投籃角度分析
> Python 籃球運動資料分析課程  
> 使用 MediaPipe 或合成範例資料計算手肘、膝蓋與肩膀角度。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 理解 pose landmarks 如何整理成表格資料。
- 由關節點計算角度時間序列。
- 輸出 CSV、角度圖與骨架示意圖。

## 完成產出
- 姿態角度 CSV、角度折線圖與骨架預覽圖。

## 課堂要求
- 按照本單元順序執行各段程式。
- 僅修改題目指定的變數、路徑或參數。
- 完成指定輸出後，記錄結果並供課堂討論。


## 課程流程
1. 使用合成資料確認資料欄位與計算流程。
2. 需要時再啟用 MediaPipe 讀取學生影片。
3. 輸出 Day 5 報告需要的資料。


In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "src" / "course_setup.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError("找不到 src/course_setup.py，請確認目前目錄位於課程 repo 內。")

from src.course_setup import DEFAULT_COURSE_ROOT, bootstrap_course_notebook  # noqa: E402

COURSE_ROOT = bootstrap_course_notebook(DEFAULT_COURSE_ROOT)


In [ ]:
# 第一次使用 MediaPipe 時再打開安裝。
# !pip install -q mediapipe

In [ ]:
import pandas as pd
from pathlib import Path
from src.video_utils import list_videos, pick_first_converted_video
from src.shooting_utils import synthetic_pose_sequence, add_pose_angles, draw_skeleton
from src.cv_utils import show_image, save_image_rgb
from src.plot_utils import plot_angle_series

converted = list_videos(COURSE_ROOT / "assets" / "converted")
print("converted videos:", [p.name for p in converted])

In [ ]:
# 本機驗證與課堂備援：沒有影片或 MediaPipe 無法執行時，使用合成 pose sequence。
use_synthetic = True
pose_df = synthetic_pose_sequence(n=80)
pose_df.head()

In [ ]:
# 選用：用 MediaPipe 讀學生影片。
# 若目前套件版本不提供 legacy solutions API，請改用教師指定版本或 MediaPipe Tasks API。
RUN_MEDIAPIPE = False

if RUN_MEDIAPIPE:
    import cv2
    import mediapipe as mp
    from typing import Any
    from src.video_utils import pick_first_converted_video

    mp_solutions: Any = getattr(mp, "solutions", None)
    if mp_solutions is None:
        raise RuntimeError(
            "目前 mediapipe 版本未提供 mediapipe.solutions.pose；"
            "請使用教師指定的 MediaPipe 版本，或改寫為 MediaPipe Tasks API。"
        )
    mp_pose: Any = mp_solutions.pose

    video_path = pick_first_converted_video(COURSE_ROOT)
    cap = cv2.VideoCapture(str(video_path))
    pose = mp_pose.Pose(static_image_mode=False, model_complexity=1)

    rows = []
    frame_idx = 0
    stride = 5
    while cap.isOpened():
        ok, frame_bgr = cap.read()
        if not ok:
            break
        if frame_idx % stride != 0:
            frame_idx += 1
            continue
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        res = pose.process(frame_rgb)
        h, w = frame_rgb.shape[:2]
        if res.pose_landmarks:
            lm = res.pose_landmarks.landmark
            # 右手側；若拍攝方向相反，可以改成 LEFT_*。
            ids = mp_pose.PoseLandmark

            def pt(name):
                p = lm[getattr(ids, name).value]
                return p.x * w, p.y * h

            shoulder = pt("RIGHT_SHOULDER")
            elbow = pt("RIGHT_ELBOW")
            wrist = pt("RIGHT_WRIST")
            hip = pt("RIGHT_HIP")
            knee = pt("RIGHT_KNEE")
            ankle = pt("RIGHT_ANKLE")
            rows.append(
                {
                    "frame": frame_idx,
                    "shoulder_x": shoulder[0],
                    "shoulder_y": shoulder[1],
                    "elbow_x": elbow[0],
                    "elbow_y": elbow[1],
                    "wrist_x": wrist[0],
                    "wrist_y": wrist[1],
                    "hip_x": hip[0],
                    "hip_y": hip[1],
                    "knee_x": knee[0],
                    "knee_y": knee[1],
                    "ankle_x": ankle[0],
                    "ankle_y": ankle[1],
                }
            )
        frame_idx += 1
    cap.release()
    pose.close()
    pose_df = add_pose_angles(pd.DataFrame(rows))
    use_synthetic = False

print("rows:", len(pose_df), "synthetic:", use_synthetic)
pose_df.head()


In [ ]:
out_csv = COURSE_ROOT / "assets" / "results" / "d4_02_pose_angles.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
pose_df.to_csv(out_csv, index=False)
print("saved:", out_csv)

plot_angle_series(
    pose_df,
    ["elbow_angle", "knee_angle", "shoulder_angle"],
    output_path=COURSE_ROOT / "assets" / "results" / "d4_02_pose_angle_plot.png",
)

In [ ]:
# 顯示其中一個 frame 的骨架圖。
row = pose_df.iloc[len(pose_df) // 2]
img = draw_skeleton(640, 480, row)
show_image(img, "pose skeleton sample")
save_image_rgb(
    COURSE_ROOT / "assets" / "results" / "d4_02_pose_skeleton_sample.png", img
)

課堂操作：若分析對象為左手投籃，MediaPipe 區塊需將 `RIGHT_*` 改為 `LEFT_*`。